# Assignment 4 — Solution


## Exercise 1 — Import the data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader.data as web

ds = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench')
df = ds[0]                                       # key 0 = monthly table
df.index = pd.to_datetime(df.index.to_timestamp())

factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
Re = df[factors]          # excess returns in percent; RF is in df['RF']

df.head()


## Exercise 2 — MVE portfolio


In [ ]:
# ── Sample moments ────────────────────────────────────────────────────────────
ERe   = Re.mean()          # monthly expected excess returns (percent)
CovRe = Re.cov()           # monthly covariance matrix (percent^2)

# ── Helper functions (reused in all subsequent exercises) ─────────────────────
def mve_weights(mu, Sigma):
    """Unit-normalised MVE weights. mu and Sigma must be numpy arrays."""
    raw = np.linalg.inv(Sigma) @ mu
    return raw / raw.sum()

def leverage_to_vol(w, Sigma, target_ann_vol):
    """Scale weights w so annualised portfolio vol equals target_ann_vol (percent)."""
    port_vol = (12 * w @ Sigma @ w) ** 0.5
    return w * target_ann_vol / port_vol

def sharpe_ratio(port_returns):
    """Annualised Sharpe Ratio from a monthly return series (in percent)."""
    return (port_returns.mean() * 12) / (port_returns.std() * 12 ** 0.5)

# ── MVE weights ────────────────────────────────────────────────────────────────
N     = len(factors)
Sigma = CovRe.values
mu    = ERe.values

w_mve = mve_weights(mu, Sigma)          # unit-normalised (sums to 1)

# Leverage to match market in-sample annualised vol
target_vol = (Re['Mkt-RF'].var() * 12) ** 0.5          # market ann. vol (%)
W_mve      = leverage_to_vol(w_mve, Sigma, target_vol)  # levered weights

# In-sample Sharpe Ratio
port_ret_mve = Re[factors].values @ W_mve
SR_mve       = sharpe_ratio(pd.Series(port_ret_mve))

print(f"Target (market) annualised vol : {target_vol:.2f}%")
print(f"MVE portfolio annualised vol   : {(12 * W_mve @ Sigma @ W_mve)**0.5:.2f}%")
print(f"In-sample Sharpe Ratio (MVE)   : {SR_mve:.3f}\n")
print("Weights:")
print(pd.Series(W_mve, index=factors).round(3))


## Exercise 3 — Choosing leverage


In [ ]:
# w_mve is the unit-normalised portfolio (sums to 1).
# All leverage variants differ only in the scalar x; composition is unchanged.

port_ERe_monthly = w_mve @ mu          # monthly E[Re] of unit portfolio (%)
port_vol_monthly = (w_mve @ Sigma @ w_mve) ** 0.5   # monthly vol (%)

# ── 3a: Target 10% annual expected excess return ───────────────────────────────
# 12 * x * port_ERe_monthly = 10
target_ERe_ann = 10.0     # percent
x_3a  = target_ERe_ann / (12 * port_ERe_monthly)
W_3a  = x_3a * w_mve

print("── 3a: target 10% annual excess return ──")
print(pd.Series(W_3a, index=factors).round(3))
print(f"Achieved annual E[Re]: {12 * W_3a @ mu:.2f}%")

# ── 3b: Target 10% annual volatility ─────────────────────────────────────────
# x * port_vol_monthly * sqrt(12) = 10
target_vol_ann = 10.0     # percent
x_3b  = target_vol_ann / (port_vol_monthly * 12 ** 0.5)
W_3b  = x_3b * w_mve

print("\n── 3b: target 10% annual volatility ──")
print(pd.Series(W_3b, index=factors).round(3))
print(f"Achieved annual vol : {(W_3b @ Sigma @ W_3b * 12)**0.5:.2f}%")

# ── 3c: Target 10% annual total return ───────────────────────────────────────
# rf_ann + 12 * x * port_ERe_monthly = 10
rf_ann = df['RF'].mean() * 12          # average annualised risk-free rate (%)
target_total_ann = 10.0
x_3c  = (target_total_ann - rf_ann) / (12 * port_ERe_monthly)
W_3c  = x_3c * w_mve

print("\n── 3c: target 10% annual total return ──")
print(f"Average annual RF used: {rf_ann:.2f}%")
print(pd.Series(W_3c, index=factors).round(3))
print(f"Achieved annual total return: {rf_ann + 12 * W_3c @ mu:.2f}%")

# ── 3d: Sharpe ratio invariance ───────────────────────────────────────────────
print("\n── 3d: Sharpe ratios across leverage choices ──")
for label, W in [("3a", W_3a), ("3b", W_3b), ("3c", W_3c), ("MVE (Ex2)", W_mve)]:
    port = pd.Series(Re[factors].values @ W)
    print(f"  SR ({label}): {sharpe_ratio(port):.4f}")


### Discussion — 3c and 3d

**3c:** Targeting a *total* return requires knowing the risk-free rate $r_f$, which is an additional
piece of information beyond what is needed for the excess-return or vol targets.
In practice, $r_f$ is directly observable (e.g. from T-bills), so this is not a hard constraint —
but it does mean the leverage depends on the level of interest rates, not just on the portfolio itself.

**3d:** Leverage does not change the Sharpe Ratio.
The scalar $x$ appears in both the numerator ($E[R^e_p] \propto x$) and the denominator
($\sigma_p \propto x$), so it cancels:

$$SR = \frac{12 \, x \, w^\top\mu}{\sqrt{12} \, x \sqrt{w^\top\Sigma w}} = \frac{\sqrt{12} \, w^\top\mu}{\sqrt{w^\top\Sigma w}}$$

The leverage choice is purely about *scale* — how much capital or risk you want to take —
not about improving the risk-adjusted return.


## Exercise 4 — Alternative allocation rules


In [ ]:
sigma_vec = Re.std().values   # sample standard deviations

# ── Rule 1: Equal Weight ───────────────────────────────────────────────────────
w_ew     = np.ones(N) / N
W_ew     = leverage_to_vol(w_ew, Sigma, target_vol)

# ── Rule 2: Risk Parity 1 (estimated Sigma, mu_i = c * sigma_i) ───────────────
# Substitute sigma_vec for ERe in the MVE formula
w_rp1    = mve_weights(sigma_vec, Sigma)
W_rp1    = leverage_to_vol(w_rp1, Sigma, target_vol)

# ── Rule 3: Risk Parity 2 (diagonal Sigma + mu_i = c * sigma_i → 1/sigma_i) ──
w_rp2_raw = 1.0 / sigma_vec
w_rp2     = w_rp2_raw / w_rp2_raw.sum()
W_rp2     = leverage_to_vol(w_rp2, Sigma, target_vol)

# ── Rule 4: Minimum Variance (mu_i = constant → Sigma^{-1} 1) ─────────────────
w_minvar  = mve_weights(np.ones(N), Sigma)
W_minvar  = leverage_to_vol(w_minvar, Sigma, target_vol)

# ── Collect ────────────────────────────────────────────────────────────────────
Weights = pd.DataFrame({
    'MVE':    W_mve,
    'EW':     W_ew,
    'RP1':    W_rp1,
    'RP2':    W_rp2,
    'MinVar': W_minvar,
}, index=factors)

print(Weights.round(3))
Weights.plot.bar(figsize=(10, 5), title='Weights by strategy')
plt.ylabel('Weight'); plt.tight_layout()


## Exercise 5 — In-sample comparison


In [ ]:
is_stats = {}
for col in Weights.columns:
    port = pd.Series(Re[factors].values @ Weights[col].values)
    is_stats[col] = sharpe_ratio(port)
is_stats['Market'] = sharpe_ratio(Re['Mkt-RF'])

sr_is = pd.Series(is_stats, name='In-sample SR')
print(sr_is.round(3))
sr_is.plot.bar(title='In-sample Sharpe Ratios', figsize=(8, 4))
plt.axhline(0, color='k', linewidth=0.5); plt.tight_layout()


### Discussion — Exercise 5

1. **MVE wins by construction.** In-sample, MVE maximises the Sharpe Ratio by definition — it is
   the portfolio that solves that exact optimisation on this data. Comparing in-sample SRs across
   strategies tells us nothing useful about which is better out-of-sample.

2. **The main problem is look-ahead bias.** The weights for every strategy were estimated using the
   same observations used to evaluate them. The MVE weights are tuned to the sample noise as well as
   the true signal, so its in-sample SR is optimistically biased.

3. **The degree of bias is largest for MVE** because it has the most free parameters (N means +
   N(N+1)/2 covariance entries). EW has zero estimation error; MinVar and RP rules lie in between.
   Out-of-sample, the ranking often reverses because simpler rules overfit less.


## Exercise 6 — Rolling out-of-sample evaluation


In [ ]:
Rp         = pd.DataFrame(dtype=float)
start_date = Re.index[len(Re) // 2]

for date in Re[start_date:].index:
    est        = Re[:date - pd.DateOffset(months=1)]
    mu_t       = est.mean().values
    Sigma_t    = est.cov().values
    sigma_t    = est.std().values
    tgt_vol_t  = (est['Mkt-RF'].var() * 12) ** 0.5
    ret_t      = Re.loc[date, factors].values

    # MVE
    Rp.at[date, 'MVE']    = leverage_to_vol(mve_weights(mu_t, Sigma_t),    Sigma_t, tgt_vol_t) @ ret_t
    # EW
    Rp.at[date, 'EW']     = leverage_to_vol(np.ones(N) / N,                Sigma_t, tgt_vol_t) @ ret_t
    # RP1
    Rp.at[date, 'RP1']    = leverage_to_vol(mve_weights(sigma_t, Sigma_t), Sigma_t, tgt_vol_t) @ ret_t
    # RP2
    w_rp2_t = (1 / sigma_t) / (1 / sigma_t).sum()
    Rp.at[date, 'RP2']    = leverage_to_vol(w_rp2_t,                        Sigma_t, tgt_vol_t) @ ret_t
    # MinVar
    Rp.at[date, 'MinVar'] = leverage_to_vol(mve_weights(np.ones(N), Sigma_t), Sigma_t, tgt_vol_t) @ ret_t

# Market benchmark over the same OOS period
Rp['Market'] = Re.loc[start_date:, 'Mkt-RF']
print(f"OOS period: {Rp.index[0].date()} → {Rp.index[-1].date()}  ({len(Rp)} months)")


## Exercise 7 — Out-of-sample performance analysis


In [ ]:
from scipy import stats

# ── Summary statistics ────────────────────────────────────────────────────────
oos = pd.DataFrame({
    'Ann. Return (%)' : Rp.mean() * 12,
    'Ann. Vol (%)'    : Rp.std() * 12 ** 0.5,
    'Sharpe Ratio'    : Rp.apply(sharpe_ratio),
})
print(oos.round(3))

# ── Cumulative returns ─────────────────────────────────────────────────────────
(1 + Rp / 100).cumprod().plot(figsize=(12, 5), title='Cumulative OOS returns')
plt.tight_layout()


In [ ]:
# ── t-tests: is each strategy's OOS return significantly different from Market? ─
print("H0: E[strategy return] = E[market return]\n")
for col in [c for c in Rp.columns if c != 'Market']:
    diff   = (Rp[col] - Rp['Market']).dropna()
    t, p   = stats.ttest_1samp(diff, 0)
    print(f"{col:8s}  mean diff = {diff.mean()*12:+.2f}%/yr  t = {t:+.2f}  p = {p:.3f}")


### Discussion — Exercise 7

1. **Is there a clear winner?**
   OOS Sharpe Ratios tend to be much closer together than in-sample, and differences are typically
   not statistically significant. MVE often no longer leads; simpler rules (EW, RP2) can match or
   beat it OOS because they do not overfit expected-return estimates.

2. **Statistical test:** A paired t-test on the monthly return difference between two strategies
   tests whether the mean difference is zero. With ~35 years of monthly data (≈420 observations) the
   test has modest power — a 1–2%/year difference in returns requires many years to be statistically
   detectable.

3. **Sources of return differences that might not interest an investor:**
   - All five strategies are **long-only combinations of long-short factors**. They can have very
     different net market exposure depending on how much `Mkt-RF` they hold. A higher return from
     more market exposure is not "skill."
   - Differences in effective leverage: even after matching vol in-sample, OOS vol can diverge,
     making raw-return comparisons misleading.
   - Transaction costs: more extreme weights (MVE can have large short-like positions within the
     factor space) imply higher rebalancing costs.

4. **Beyond Sharpe Ratios:** drawdown analysis, turnover, tail risk (CVaR), performance in
   crisis periods (2001, 2008, 2020), and the information ratio relative to the market are all
   important complements to a single summary statistic.
